# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install matplotlib pandas --quiet

**Imports**

In [ ]:
# --- Imports: libraries used in this notebook ---
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

**Declarations**

Set the variables below and run the generation cell. 

- The first data array will be taken from `./data1.csv` relative to this notebook.
- The second data array will be taken from `./data2.csv` relative to this notebook.


In [ ]:
# --- Declarations: editable variables for students ---
csv_file_1 = ...
csv_file_2 = ...

In [ ]:
# --- Load data from CSV files ---
p1 = Path(csv_file_1)
p2 = Path(csv_file_2)

# Load first column from each CSV (works whether column is named or not)
df1 = pd.read_csv(p1)
df2 = pd.read_csv(p2)

if df1.shape[1] < 1 or df2.shape[1] < 1:
    raise ValueError('One of the CSV files contains no columns')

data_01 = df1.iloc[:, 0].to_numpy()
data_02 = df2.iloc[:, 0].to_numpy()

# Ensure lengths match for correlation operations
if len(data_01) != len(data_02):
    raise ValueError(f'Input files have different lengths: {len(data_01)} vs {len(data_02)}')

print(f'Loaded {p1.name} and {p2.name} (N={len(data_01)})')

### 1. Cross-Correlation 
Cross-correlation measures the similarity between two signals (or sequences) as a function of the time-lag applied to one of them. It is widely used in signal processing to find the degree to which one signal correlates with another, possibly shifted in time.  

#### 1.1 Definition

Cross-correlation of two discrete signals (or vectors) $ x $ and $ y $ is defined as:

$$
r_{xy}[k] = \sum_{n=-\infty}^{\infty} x[n] \cdot y[n+k],
$$

where
- $ r_{xy}[k] $ is the cross-correlation at lag $ k $,
- $ x[n] $ and $ y[n] $ are the $n$-th value of the input signals (or vectors),
- $ k $ is the lag, or the amount by which the second signal $ y $ is shifted.

For finite-length vectors $ x[n] $ and $ y[n] $, the summation limits are constrained by the length of the vectors, so the cross-correlation formula becomes:

$$
r_{xy}[k] = \sum_{n=0}^{N-1} x[n] \cdot y[n+k],
$$

where
- $ N $ is the length of the vectors $ x $ and $ y $,
- $ k $ is the lag, ranging from $ -N+1 $ to $ N-1 $.





In [ ]:
# --- Compute cross-correlation ---
cross_corr_random = np.correlate(data_01, data_02, mode='full')

In [ ]:
# --- Plot cross-correlation ---
fig, ax_cross = plt.subplots(figsize=(8, 5))
theLineWidth = 3

ax_cross.plot(cross_corr_random, linewidth=theLineWidth, color='c',
              label='cross correlation of data_01 and data_02')

ax_cross.set_xlabel('n')
ax_cross.set_ylabel('correlation value')
ax_cross.grid(True)
ax_cross.set_facecolor((.2, 0.2, 0.3))
ax_cross.legend(fancybox=True, framealpha=1,
                loc="lower left", facecolor=(.6, 0.61, 0.8))
plt.tight_layout()

### 2. Autocorrelation
Autocorrelation, on the other hand, measures the correlation of a signal with a delayed copy of itself. It helps determine repeating patterns or periodicity within a signal.

#### 2.1 Definition
The **autocorrelation** of a signal $ x[n] $ (correlating the signal with itself) is a special case of cross-correlation, where $ y[n] = x[n] $. The autocorrelation function is defined as:

$$
r_{xx}[k] = \sum_{n=-\infty}^{\infty} x[n] \cdot x[n+k]
$$

For finite-length vectors $ x[n] $ of length $ N $, the autocorrelation is computed as:

$$
r_{xx}[k] = \sum_{n=0}^{N-1-k} x[n] \cdot x[n+k],
$$

where
- $ r_{xx}[k] $ is the autocorrelation at lag $ k $,
- $ k $ is the lag, ranging from $ 0 $ to $ N-1 $.

In this case, autocorrelation is symmetric about $ k = 0 $, meaning $ r_{xx}[k] = r_{xx}[-k] $.

In [ ]:
# --- Compute autocorrelation ---
auto_corr_01 = np.correlate(data_01, data_01, mode='full')
auto_corr_02 = np.correlate(data_02, data_02, mode='full')

In [ ]:
# --- Plot autocorrelation ---
fig, ax_auto = plt.subplots(figsize=(8, 5))

ax_auto.plot(auto_corr_01, linewidth=theLineWidth,
             color='m', label='auto correlation of data_01 (random)')
ax_auto.plot(auto_corr_02, linewidth=theLineWidth,
             color='y', label='auto correlation of data_02 (constant 1)')

ax_auto.set_xlabel('n')
ax_auto.set_ylabel('correlation value')
ax_auto.grid(True)
ax_auto.set_facecolor((.2, 0.2, 0.3))
ax_auto.legend(fancybox=True, framealpha=1,
               loc="lower left", facecolor=(.6, 0.61, 0.8))
plt.tight_layout()

#### 2.2 Power of Data Arrays
A special case of autocorrelation occurs at lag $k=0$, where the autocorrelation function reduces to the signal energy. When we average over the length of the array, we find the signal power.

Power in a real signal (or data array) is the average of its squared values, showing how strong the signal is over time. It gives an estimate of the average energy over all the samples. 
For discrete signals, this is called the mean square value.

Signal power is an important measure used in signal processing, communications, and data analysis.

##### Definition

Mathematically, for a discrete signal (or data array) $ x[n] $, consisting of $ N $ samples, the power $ P $ of the signal is defined as:

$$
P = \frac{r_{xx}[0]}{N} , \qquad r_{xx}[0] = \sum_{n=0}^{N-1-k} x[n]^2,
$$

where
- $ x[n] $ is the value of the signal at sample $ n $,
- $ N $ is the total number of samples in the array.


In [ ]:
# --- Power utility: function to compute mean-square (power) of an array ---
def calc_power(data):
    squared_data = np.square(data)
    sum_of_squares = np.sum(squared_data)
    power = sum_of_squares / len(data)
    return power


print("Power of data_01:   %5.3f%%" % (calc_power(data_01) * 100))
print("Power of data_02:   %5.3f%%" % (calc_power(data_02) * 100))

##### Zero-Mean Signals and Variance

When a signal’s mean value is removed, we obtain a zero-mean signal, which represents only the fluctuations around its average.
For such signals, the variance equals the average power.


In [ ]:
# --- Zero-mean utility: subtract mean from data ---
def zero_mean(data):
    return data - np.mean(data)

# Original and zero-meaned data
data_01_zm = zero_mean(data_01)
data_02_zm = zero_mean(data_02)

print("Power of zero-mean data_01:   %5.3f%%" % (calc_power(data_01_zm) * 100))
print("Power of zero-mean data_02:   %5.3f%%" % (calc_power(data_02_zm) * 100))